# Laya alert fine-tune

Fine-tunes Laya on one yes/no question: is this job worth an alert for the candidate? See
`docs/specs/2026-10-04-laya-alert-finetune-design.md`. This notebook only calls functions in
`training/laya_train.py` — the logic itself is unit-tested on a laptop; this notebook just runs it on a GPU
and exports the result.

**Upload folder** (you will be prompted in the next cell): `laya_train.py`, `posting.py`, `scoring_resume.txt`,
`pool_old.jsonl`, `pool_jobhunter.jsonl`, `heldout_ids.txt`.

**Runtime: A100** (Runtime > Change runtime type > A100 GPU) before running the cells below.


In [ ]:
!pip -q install laya==0.3.20 scikit-learn pandas
from google.colab import drive, files
drive.mount('/content/drive')
import sys, json, torch
from pathlib import Path
UP = Path('/content/upload'); UP.mkdir(exist_ok=True)
print('Upload: laya_train.py, posting.py, scoring_resume.txt, pool_old.jsonl, pool_jobhunter.jsonl, heldout_ids.txt')
for name, data in files.upload().items():
    (UP / name).write_bytes(data)
sys.path.insert(0, str(UP))
import laya_train as lt

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type > A100 GPU, then rerun this cell.')
device = torch.device('cuda')
if torch.cuda.get_device_properties(0).total_memory < 70e9:
    settings = lt.Settings(micro_batch=4, grad_accum=16)  # effective batch stays 64, less GPU memory per step
    print(f'GPU memory {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB (<70 GB): using '
          'micro_batch=4, grad_accum=16')
else:
    settings = lt.Settings()
    print(f'GPU memory {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB: using default settings '
          '(micro_batch=8, grad_accum=8)')

SAFE = Path('/content/drive/MyDrive/laya_alert/inprogress')


def save_best_so_far(b):
    """Persist the best epoch's weights to Drive as training goes, so a Colab disconnect does not lose it."""
    SAFE.mkdir(parents=True, exist_ok=True)
    torch.save(b.state, SAFE / 'best.pt')


In [ ]:
from huggingface_hub import snapshot_download
base = snapshot_download('convaiinnovations/laya',
                         allow_patterns=['rl_agent_config.json', 'model.safetensors', 'tokenizer/*', 'encoder/*'])
held = set((UP / 'heldout_ids.txt').read_text().split())
rows = [r for r in lt.read_pool(UP / 'pool_old.jsonl', 'old') + lt.read_pool(UP / 'pool_jobhunter.jsonl', 'jobhunter')
        if r['job_id'] not in held]
splits = lt.split_rows(rows, settings)
train_rows = lt.sample_train(splits['train'], settings)
resume = (UP / 'scoring_resume.txt').read_text().strip()
model, tok, base_cfg = lt.load_base(base, device)
items = {name: lt.build_items(rs, resume, tok, settings) for name, rs in
         (('train', train_rows), ('select', splits['select']), ('calibrate', splits['calibrate']))}
health = {name: lt.input_health(rs, items[name]) for name, rs in
          (('train', train_rows), ('select', splits['select']), ('calibrate', splits['calibrate']))}
for name, rs in (('train', train_rows), ('select', splits['select']), ('calibrate', splits['calibrate'])):
    from collections import Counter; print(name, len(rs), dict(Counter(r['label'] for r in rs)), health[name])


In [ ]:
result = lt.train(model, items['train'], items['select'], settings, device, on_best=save_best_so_far,
                  pad_id=tok.pad_token_id)
model.load_state_dict(result.state)


**If Colab disconnected after at least one epoch:** run this cell to restore the best epoch from Drive, then continue from the calibration cell below. This is optional — skip it if training finished normally.

In [ ]:
# Optional: only needed if Colab disconnected mid-training. Restores the best epoch saved to Drive by
# save_best_so_far above, so you can continue from the calibration cell below without retraining.
best_path = SAFE / 'best.pt'
if 'result' in globals():
    print('Training finished in this session: nothing to restore.')
elif best_path.exists():
    state = torch.load(best_path, map_location=device)
    model.load_state_dict(state)
    result = lt.TrainResult(state, epoch=-1, cutoff=None, history=[])
    print('Restored the best saved epoch from', best_path)
else:
    print('No saved checkpoint found at', best_path, '- nothing to restore; run the training cell above.')


In [ ]:
import numpy as np
from laya import ece_score
cal_logits = lt.predict_logits(model, items['calibrate'], settings, device, tok.pad_token_id)
cal_y = np.array([it['label'] for it in items['calibrate']])
temperature = lt.fit_temperature(cal_logits, cal_y)
sel_logits = lt.predict_logits(model, items['select'], settings, device, tok.pad_token_id)
sel_y = np.array([it['label'] for it in items['select']])
gold = [it['gold'] for it in items['select']]
p = lt.probs_from_logits(sel_logits, temperature)
cut = lt.pick_alert_cutoff(p, sel_y, settings.min_recall)
save_at = lt.pick_save_cutoff(p, gold, cut.threshold)


def ece(probs, y):
    conf = np.maximum(probs, 1 - probs)
    return ece_score(conf, (probs >= 0.5) == (y == 1))


decided = np.where(p >= cut.threshold, 'alert', np.where(p >= save_at, 'save', 'skip'))
confusion = {g: {d: int(((np.array(gold) == g) & (decided == d)).sum()) for d in ('alert', 'save', 'skip')} for g in lt.LABELS}
report = {'select': {'alerts': cut.alerts, 'precision': cut.precision, 'recall': cut.recall, 'met_rule': cut.met_rule},
          'alert_at': cut.threshold, 'save_at': save_at, 'temperature': temperature,
          'base_rate': float(sel_y.mean()),
          'ece_before': ece(lt.probs_from_logits(sel_logits), sel_y), 'ece_after': ece(p, sel_y),
          'table': lt.cutoff_table(p, sel_y, cut.threshold), 'p_min': float(p.min()), 'p_max': float(p.max()),
          'confusion': confusion, 'health': health, 'history': result.history, 'best_epoch': result.epoch}
print(lt.report_text(report))


**Before you export:** read the report printed above. If any line starts with `!!`, do **not** switch to this
model — instead, send the report to Claude Code so it can look into why.


In [ ]:
OUT = Path('/content/drive/MyDrive/laya_alert/laya_alert_v5')
lt.export_checkpoint(result.state, base, base_cfg, OUT, settings, lt.ALERT_QUESTION, cut.threshold, save_at,
                     temperature, report)
report_txt = lt.report_text(report)
(OUT / 'report.txt').write_text(report_txt)
print(report_txt)
print('Saved to', OUT, '- download this folder into the models folder.')
